In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, Subset
import copy
from tqdm import tqdm

In [2]:
# --- Image Encoder ---
class SimpleImageEncoder(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        resnet = models.resnet50(pretrained=True)
        modules = list(resnet.children())[:-1]  # remove classification head
        self.resnet = nn.Sequential(*modules)
        self.fc = nn.Linear(resnet.fc.in_features, output_dim)
        
    def forward(self, x):
        x = self.resnet(x)            # (B, features, 1, 1)
        x = x.view(x.size(0), -1)       # (B, features)
        x = self.fc(x)                # (B, output_dim)
        return x

# --- Text Encoder ---
class SimpleTextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
    def forward(self, x):
        x = self.embedding(x)         # (B, seq_len, embed_dim)
        _, h = self.rnn(x)            # h: (1, B, hidden_dim)
        return h.squeeze(0)           # (B, hidden_dim)

# --- Overall CLIP Model ---
class SimpleCLIP(nn.Module):
    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()
        self.image_encoder = SimpleImageEncoder(output_dim=embed_dim)
        self.text_encoder = SimpleTextEncoder(vocab_size, embed_dim=embed_dim, hidden_dim=embed_dim)
        self.temperature = 0.07
        
    def forward(self, image, text):
        image_features = self.image_encoder(image)  # (B, embed_dim)
        text_features = self.text_encoder(text)       # (B, embed_dim)
        image_features = F.normalize(image_features, p=2, dim=-1)
        text_features = F.normalize(text_features, p=2, dim=-1)
        return image_features, text_features
    
    def compute_loss(self, image, text):
        image_features, text_features = self.forward(image, text)
        logits = image_features @ text_features.t() / self.temperature  # (B, B)
        labels = torch.arange(image_features.size(0)).to(image_features.device)
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.t(), labels)
        loss = (loss_i2t + loss_t2i) / 2
        return loss
    
    def analyze_performance(self, image, text):
        image_features, text_features = self.forward(image, text)
        similarity = image_features @ text_features.t()  # (B, B)
        batch_size = similarity.size(0)
        ranks = []
        for i in range(batch_size):
            sorted_sim, indices = similarity[i].sort(descending=True)
            rank = (indices == i).nonzero(as_tuple=False).item() + 1
            ranks.append(rank)
        avg_rank = sum(ranks) / len(ranks)
        return avg_rank, ranks

# --- Collate Function ---
def collate_fn(batch, seq_len=16):
    images = []
    texts = []
    for image, label in batch:
        images.append(image)
        # Text: first token = label+1, rest padded with 0.
        text_tensor = torch.zeros(seq_len, dtype=torch.long)
        text_tensor[0] = label + 1
        texts.append(text_tensor)
    images = torch.stack(images)
    texts = torch.stack(texts)
    return images, texts

# --- Training Function ---
def train_model(model, data_loader, optimizer, device, epochs=1):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        pbar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, texts in pbar:
            images, texts = images.to(device), texts.to(device)
            optimizer.zero_grad()
            loss = model.compute_loss(images, texts)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=running_loss/((pbar.n + 1)))
    avg_loss = running_loss / len(data_loader)
    return model, avg_loss

def evaluate_model(model, data_loader, device):
    model.eval()
    total_loss = 0.0
    all_ranks = []
    with torch.no_grad():
        for images, texts in data_loader:
            images, texts = images.to(device), texts.to(device)
            loss = model.compute_loss(images, texts)
            total_loss += loss.item()
            avg_rank, _ = model.analyze_performance(images, texts)
            all_ranks.append(avg_rank)
    return total_loss / len(data_loader), sum(all_ranks) / len(all_ranks)

In [3]:
# --- Main Script ---
if __name__ == "__main__":
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # device = "cpu"
    print("Using device:", device)
    
    # ------------------ Scenario 1: Big Dataset (CIFAR10) ------------------
    # CIFAR10: 10 classes -> vocab_size = 10 + 1 = 11
    batch_size = 16
    seq_len = 16
    vocab_cifar = 11
    epochs = 3  # for demonstration
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    
    print("\n--- Loading CIFAR10 (Big Dataset) ---")
    cifar_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    cifar_test  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    cifar_train_loader = DataLoader(cifar_train, batch_size=batch_size, shuffle=True,
                                    collate_fn=lambda b: collate_fn(b, seq_len))
    cifar_test_loader  = DataLoader(cifar_test, batch_size=batch_size, shuffle=False,
                                    collate_fn=lambda b: collate_fn(b, seq_len))
    
    print("\nScenario 1: Training from scratch on CIFAR10")
    model_cifar = SimpleCLIP(vocab_size=vocab_cifar, embed_dim=256).to(device)
    optimizer_cifar = torch.optim.Adam(model_cifar.parameters(), lr=1e-4)
    model_cifar, train_loss_cifar = train_model(model_cifar, cifar_train_loader, optimizer_cifar, device, epochs=epochs)
    test_loss_cifar, avg_rank_cifar = evaluate_model(model_cifar, cifar_test_loader, device)
    print(f"CIFAR10 - Train Loss: {train_loss_cifar:.4f}, Test Loss: {test_loss_cifar:.4f}, Average Rank: {avg_rank_cifar:.4f}")
    
    # ------------------ Scenario 2: Small Dataset (Food101 Subset) ------------------
    # Food101: 101 classes -> vocab_size = 101 + 1 = 102
    vocab_food = 102
    print("\n--- Loading Food101 (Small Dataset) ---")
    # Food101 available via torchvision.datasets.Food101
    try:
        from torchvision.datasets import Food101
    except ImportError:
        raise ImportError("Make sure torchvision>=0.12 is installed to use Food101.")
    
    food_train_full = Food101(root='./data', split='train', download=True, transform=transform)
    food_test = Food101(root='./data', split='test', download=True, transform=transform)
    # Use 10% of the Food101 training set as the small dataset
    small_food_size = int(0.1 * len(food_train_full))
    food_train_small = Subset(food_train_full, range(small_food_size))
    food_train_loader = DataLoader(food_train_small, batch_size=batch_size, shuffle=True,
                                   collate_fn=lambda b: collate_fn(b, seq_len))
    food_test_loader = DataLoader(food_test, batch_size=batch_size, shuffle=False,
                                  collate_fn=lambda b: collate_fn(b, seq_len))
    
    print("\nScenario 2: Training from scratch on Food101 Subset")
    model_food = SimpleCLIP(vocab_size=vocab_food, embed_dim=256).to(device)
    optimizer_food = torch.optim.Adam(model_food.parameters(), lr=1e-4)
    model_food, train_loss_food = train_model(model_food, food_train_loader, optimizer_food, device, epochs=epochs)
    test_loss_food, avg_rank_food = evaluate_model(model_food, food_test_loader, device)
    print(f"Food101 Subset - Train Loss: {train_loss_food:.4f}, Test Loss: {test_loss_food:.4f}, Average Rank: {avg_rank_food:.4f}")
    
    # ------------------ Scenario 3: Transfer Learning (Pretrain on CIFAR10 -> Fine-tune on Food101 Subset) ------------------
    print("\nScenario 3: Transfer Learning (CIFAR10 -> Food101 Subset)")
    # Initialize a new Food101 model
    model_transfer = SimpleCLIP(vocab_size=vocab_food, embed_dim=256).to(device)
    # Transfer the image encoder weights from the CIFAR10 model (they are domain-agnostic)
    model_transfer.image_encoder.load_state_dict(model_cifar.image_encoder.state_dict())
    # (The text encoder is reinitialized to handle 102 tokens)
    optimizer_transfer = torch.optim.Adam(model_transfer.parameters(), lr=1e-4)
    model_transfer, train_loss_transfer = train_model(model_transfer, food_train_loader, optimizer_transfer, device, epochs=epochs)
    test_loss_transfer, avg_rank_transfer = evaluate_model(model_transfer, food_test_loader, device)
    print(f"Transfer Learning - Additional Train Loss: {train_loss_transfer:.4f}, Test Loss: {test_loss_transfer:.4f}, Average Rank: {avg_rank_transfer:.4f}")
    
    # ------------------ Summary ------------------
    print("\n--- Summary of Results ---")
    print(f"1. CIFAR10 Model: Test Loss = {test_loss_cifar:.4f}, Average Rank = {avg_rank_cifar:.4f}")
    print(f"2. Food101 Model (from scratch): Test Loss = {test_loss_food:.4f}, Average Rank = {avg_rank_food:.4f}")
    print(f"3. Transfer Learning (CIFAR10 -> Food101): Test Loss = {test_loss_transfer:.4f}, Average Rank = {avg_rank_transfer:.4f}")

Using device: cuda

--- Loading CIFAR10 (Big Dataset) ---

Scenario 1: Training from scratch on CIFAR10


/home/akshal/Projects/clip-model-test/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/akshal/Projects/clip-model-test/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/3:   0%|          | 0/3125 [00:00<?, ?it/s]/home/akshal/Projects/clip-model-test/venv/lib/python3.12/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at /pytorch/aten/src/ATen/Co

CIFAR10 - Train Loss: 0.9330, Test Loss: 0.9678, Average Rank: 1.9039

--- Loading Food101 (Small Dataset) ---


100%|██████████| 5.00G/5.00G [16:08<00:00, 5.16MB/s]   



Scenario 2: Training from scratch on Food101 Subset


Epoch 3/3: 100%|██████████| 474/474 [01:26<00:00,  5.49it/s, loss=0.971]


Food101 Subset - Train Loss: 0.9708, Test Loss: 3.0454, Average Rank: 8.4818

Scenario 3: Transfer Learning (CIFAR10 -> Food101 Subset)


Epoch 3/3: 100%|██████████| 474/474 [01:25<00:00,  5.55it/s, loss=1.03]


Transfer Learning - Additional Train Loss: 1.0333, Test Loss: 3.0204, Average Rank: 8.4812

--- Summary of Results ---
1. CIFAR10 Model: Test Loss = 0.9678, Average Rank = 1.9039
2. Food101 Model (from scratch): Test Loss = 3.0454, Average Rank = 8.4818
3. Transfer Learning (CIFAR10 -> Food101): Test Loss = 3.0204, Average Rank = 8.4812
